<a href="https://colab.research.google.com/github/AndiSyafarIlham/online-retail-analysis/blob/main/notebooks/02_sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================
# CELL 1: MUAT DATA BERSIH KE DATABASE SQLITE
# =====================================================
import pandas as pd
import sqlite3

url = "https://github.com/AndiSyafarIlham/online-retail-analysis/raw/main/data/online_retail_clean.csv.gz"
df = pd.read_csv(url, parse_dates=["InvoiceDate"])
df["InvoiceDate"] = df["InvoiceDate"].astype(str)   # SQLite menyimpan tanggal sebagai teks

con = sqlite3.connect("retail.db")
df.to_sql("sales", con, if_exists="replace", index=False)

def sql(query):
    return pd.read_sql_query(query, con)

print("Baris di tabel sales:", sql("SELECT COUNT(*) AS n FROM sales").iloc[0, 0])
sql("SELECT * FROM sales LIMIT 5")

Baris di tabel sales: 522710


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Bulan,Hari,Jam,punya_customer_id
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12,Wednesday,8,1
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8,1
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12,Wednesday,8,1
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8,1
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,8,1


In [2]:
# =====================================================
# PERTANYAAN 1: TREN PENJUALAN PER BULAN
# =====================================================
q1 = """
WITH bulanan AS (
    SELECT
        Bulan,
        ROUND(SUM(Revenue), 0)       AS revenue,
        COUNT(DISTINCT InvoiceNo)    AS jml_order
    FROM sales
    GROUP BY Bulan
)
SELECT
    Bulan,
    revenue,
    jml_order,
    ROUND((revenue - LAG(revenue) OVER (ORDER BY Bulan)) * 100.0
          / LAG(revenue) OVER (ORDER BY Bulan), 1) AS growth_pct
FROM bulanan
ORDER BY Bulan
"""
hasil_q1 = sql(q1)
hasil_q1

,Bulan,revenue,jml_order,growth_pct
0,2010-12,776318.0,1551,NaN
1,2011-01,670639.0,1081,-13.6
2,2011-02,508082.0,1093,-24.2
3,2011-03,690592.0,1440,35.9
4,2011-04,515870.0,1235,-25.3
5,2011-05,740472.0,1668,43.5
6,2011-06,738234.0,1525,-0.3
7,2011-07,688803.0,1452,-6.7
8,2011-08,724708.0,1340,5.2
9,2011-09,1029245.0,1819,42.0


In [3]:
# =====================================================
# CELL 3: MUAT DATA RETUR + VIEW RETUR BERSIH
# =====================================================
url_cancel = "https://github.com/AndiSyafarIlham/online-retail-analysis/raw/main/data/online_retail_cancel.csv"
df_cancel = pd.read_csv(url_cancel)
df_cancel.to_sql("cancel", con, if_exists="replace", index=False)

con.executescript("""
DROP VIEW IF EXISTS retur;
CREATE VIEW retur AS
SELECT
    *,
    UPPER(TRIM(Description)) AS Desc_up,
    -Quantity                AS qty_retur,
    -Quantity * UnitPrice    AS nilai_retur
FROM cancel
WHERE StockCode NOT IN ('POST','D','M','DOT','CRUK','BANK CHARGES','AMAZONFEE','S','B','PADS');
""")

sql("""
SELECT COUNT(*) AS baris_retur,
       COUNT(DISTINCT InvoiceNo) AS invoice_batal,
       SUM(qty_retur) AS total_unit_retur,
       ROUND(SUM(nilai_retur), 0) AS nilai_retur
FROM retur
""")

,baris_retur,invoice_batal,total_unit_retur,nilai_retur
0,8706,3422,272032,478789.0


In [4]:
q2a = """
SELECT
    Country,
    ROUND(SUM(Revenue), 0)                                      AS revenue,
    COUNT(DISTINCT InvoiceNo)                                   AS jml_order,
    ROUND(SUM(Revenue) * 100.0 / (SELECT SUM(Revenue) FROM sales), 2) AS pct_revenue
FROM sales
GROUP BY Country
ORDER BY revenue DESC
LIMIT 10
"""
hasil_q2a = sql(q2a)
hasil_q2a

,Country,revenue,jml_order,pct_revenue
0,United Kingdom,8726607.0,17901,85.10
1,Netherlands,283889.0,93,2.77
2,EIRE,276091.0,284,2.69
3,Germany,205381.0,443,2.00
4,France,184679.0,383,1.80
5,Australia,138104.0,56,1.35
6,Spain,55707.0,88,0.54
7,Switzerland,53066.0,50,0.52
8,Japan,37416.0,19,0.36
9,Belgium,36927.0,98,0.36


In [5]:
q2b = """
WITH jual AS (
    SELECT Description AS produk,
           SUM(Quantity) AS qty_jual,
           SUM(Revenue)  AS rev_kotor
    FROM sales
    GROUP BY Description
),
ret AS (
    SELECT Desc_up AS produk,
           SUM(qty_retur)   AS qty_retur,
           SUM(nilai_retur) AS nilai_retur
    FROM retur
    GROUP BY Desc_up
)
SELECT
    j.produk,
    j.qty_jual,
    ROUND(j.rev_kotor, 0)                              AS rev_kotor,
    COALESCE(r.qty_retur, 0)                           AS qty_retur,
    ROUND(COALESCE(r.nilai_retur, 0), 0)               AS nilai_retur,
    ROUND(j.rev_kotor - COALESCE(r.nilai_retur, 0), 0) AS rev_bersih
FROM jual j
LEFT JOIN ret r ON j.produk = r.produk
ORDER BY rev_bersih DESC
LIMIT 10
"""
hasil_q2b = sql(q2b)
hasil_q2b

,produk,qty_jual,rev_kotor,qty_retur,nilai_retur,rev_bersih
0,REGENCY CAKESTAND 3 TIER,13851,174157.0,857,9723.0,164434.0
1,WHITE HANGING HEART T-LIGHT HOLDER,37872,106237.0,2578,6624.0,99612.0
2,PARTY BUNTING,18283,99445.0,277,1201.0,98244.0
3,JUMBO BAG RED RETROSPOT,48371,94160.0,1115,1984.0,92176.0
4,RABBIT NIGHT LIGHT,30739,66870.0,108,208.0,66662.0
5,PAPER CHAIN KIT 50'S CHRISTMAS,19329,64876.0,453,1160.0,63715.0
6,ASSORTED COLOUR BIRD ORNAMENT,36362,58928.0,80,135.0,58792.0
7,CHILLI LIGHTS,10302,54096.0,80,350.0,53747.0
8,SPOTTY BUNTING,8320,42513.0,110,483.0,42031.0
9,JUMBO BAG PINK POLKADOT,21448,42401.0,456,817.0,41584.0


In [6]:
# Total retur sebagai % dari revenue
print(sql("""
SELECT ROUND((SELECT SUM(nilai_retur) FROM retur) * 100.0
             / (SELECT SUM(Revenue) FROM sales), 2) AS retur_pct_dari_revenue
"""))

# Produk dengan persentase retur tertinggi (minimal terjual 500 unit)
q4 = """
WITH jual AS (
    SELECT Description AS produk, SUM(Quantity) AS qty_jual
    FROM sales GROUP BY Description
    HAVING SUM(Quantity) >= 500
),
ret AS (
    SELECT Desc_up AS produk, SUM(qty_retur) AS qty_retur, COUNT(*) AS kali_retur
    FROM retur GROUP BY Desc_up
)
SELECT j.produk, j.qty_jual, r.qty_retur,
       ROUND(r.qty_retur * 100.0 / j.qty_jual, 1) AS retur_pct,
       r.kali_retur
FROM jual j JOIN ret r ON j.produk = r.produk
ORDER BY retur_pct DESC
LIMIT 8
"""
hasil_q4 = sql(q4)
hasil_q4

   retur_pct_dari_revenue
0                    4.67


,produk,qty_jual,qty_retur,retur_pct,kali_retur
0,"PAPER CRAFT , LITTLE BIRDIE",80995,80995,100.0,1
1,ROTATING SILVER ANGELS T-LIGHT HLDR,9461,9376,99.1,4
2,MEDIUM CERAMIC TOP STORAGE JAR,78033,74494,95.5,10
3,PANTRY CHOPPING BOARD,1154,946,82.0,6
4,IVORY CHANDELIER T-LIGHT HOLDER,657,432,65.8,5
5,BEADED CHANDELIER T-LIGHT HOLDER,695,433,62.3,6
6,FLOWERS CHANDELIER T-LIGHT HOLDER,960,579,60.3,7
7,TEA TIME PARTY BUNTING,2578,1424,55.2,7


In [7]:
# =====================================================
# CELL 7: SEGMENTASI PELANGGAN (RFM)
# =====================================================
con.executescript("""
DROP VIEW IF EXISTS rfm_base;
DROP VIEW IF EXISTS rfm_skor;
DROP VIEW IF EXISTS rfm_segmen;

-- 1. Hitung Recency, Frequency, Monetary per pelanggan
CREATE VIEW rfm_base AS
SELECT
    CustomerID,
    CAST(julianday((SELECT DATE(MAX(InvoiceDate), '+1 day') FROM sales))
         - julianday(DATE(MAX(InvoiceDate))) AS INTEGER) AS recency,
    COUNT(DISTINCT InvoiceNo)  AS frequency,
    ROUND(SUM(Revenue), 2)     AS monetary
FROM sales
WHERE punya_customer_id = 1
GROUP BY CustomerID;

-- 2. Beri skor 1-5 (5 = terbaik)
CREATE VIEW rfm_skor AS
SELECT *,
    NTILE(5) OVER (ORDER BY recency DESC)           AS r_skor,
    NTILE(5) OVER (ORDER BY frequency, monetary)    AS f_skor,
    NTILE(5) OVER (ORDER BY monetary)               AS m_skor
FROM rfm_base;

-- 3. Tentukan segmen
CREATE VIEW rfm_segmen AS
SELECT *,
    CASE
        WHEN r_skor >= 4 AND f_skor >= 4 THEN 'Champions'
        WHEN r_skor >= 3 AND f_skor >= 3 THEN 'Loyal'
        WHEN r_skor >= 4 AND f_skor <= 2 THEN 'Pelanggan Baru'
        WHEN r_skor <= 2 AND f_skor >= 3 THEN 'Berisiko Pergi'
        WHEN r_skor <= 2 AND f_skor <= 2 THEN 'Hilang'
        ELSE 'Perlu Perhatian'
    END AS segmen
FROM rfm_skor;
""")

sql("""
SELECT COUNT(*) AS pelanggan,
       MIN(recency) AS recency_min, MAX(recency) AS recency_max,
       MIN(frequency) AS freq_min, MAX(frequency) AS freq_max,
       ROUND(SUM(monetary), 0) AS total_revenue
FROM rfm_base
""")

,pelanggan,recency_min,recency_max,freq_min,freq_max,total_revenue
0,4334,1,374,1,206,8743914.0


In [8]:
q3 = """
SELECT
    segmen,
    COUNT(*)                                                       AS pelanggan,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rfm_segmen), 1) AS pct_pelanggan,
    ROUND(SUM(monetary), 0)                                        AS revenue,
    ROUND(SUM(monetary) * 100.0 / (SELECT SUM(monetary) FROM rfm_segmen), 1) AS pct_revenue,
    ROUND(AVG(recency), 0)                                         AS rata2_recency,
    ROUND(AVG(frequency), 1)                                       AS rata2_frekuensi
FROM rfm_segmen
GROUP BY segmen
ORDER BY revenue DESC
"""
hasil_q3 = sql(q3)
hasil_q3

,segmen,pelanggan,pct_pelanggan,revenue,pct_revenue,rata2_recency,rata2_frekuensi
0,Champions,1123,25.9,5849479.0,66.9,14.0,10.0
1,Loyal,829,19.1,1349787.0,15.4,38.0,3.6
2,Berisiko Pergi,648,15.0,862296.0,9.9,149.0,3.4
3,Hilang,1086,25.1,457841.0,5.2,218.0,1.1
4,Perlu Perhatian,339,7.8,131960.0,1.5,52.0,1.1
5,Pelanggan Baru,309,7.1,92551.0,1.1,18.0,1.2


In [9]:
# =====================================================
# CELL 9: EKSPOR HASIL ANALISIS UNTUK DASHBOARD
# =====================================================
import os
from google.colab import files

os.makedirs("export", exist_ok=True)

def simpan(nama, query):
    hasil = sql(query)
    hasil.to_csv(f"export/{nama}.csv", index=False)
    print(f"{nama:20s} {len(hasil):6,d} baris")
    return hasil

# 1. KPI utama (angka besar di bagian atas dashboard)
simpan("kpi_ringkasan", """
SELECT
    ROUND(SUM(Revenue), 2)                          AS total_revenue,
    COUNT(DISTINCT InvoiceNo)                       AS jumlah_order,
    COUNT(DISTINCT CustomerID)                      AS jumlah_pelanggan,
    ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS rata2_nilai_order,
    ROUND((SELECT SUM(nilai_retur) FROM retur), 2)  AS nilai_retur
FROM sales
""")

# 2. Tren bulanan
simpan("tren_bulanan", """
SELECT Bulan,
       ROUND(SUM(Revenue), 0)       AS revenue,
       COUNT(DISTINCT InvoiceNo)    AS jml_order,
       COUNT(DISTINCT CustomerID)   AS jml_pelanggan
FROM sales
GROUP BY Bulan
ORDER BY Bulan
""")

# 3. Revenue per negara (semua negara)
simpan("revenue_negara", """
SELECT Country,
       ROUND(SUM(Revenue), 0)    AS revenue,
       COUNT(DISTINCT InvoiceNo) AS jml_order
FROM sales
GROUP BY Country
ORDER BY revenue DESC
""")

# 4. Produk teratas (revenue bersih setelah retur)
simpan("produk_bersih", """
WITH jual AS (
    SELECT Description AS produk, SUM(Quantity) AS qty_jual, SUM(Revenue) AS rev_kotor
    FROM sales GROUP BY Description
),
ret AS (
    SELECT Desc_up AS produk, SUM(qty_retur) AS qty_retur, SUM(nilai_retur) AS nilai_retur
    FROM retur GROUP BY Desc_up
)
SELECT j.produk, j.qty_jual,
       ROUND(j.rev_kotor, 0)                              AS rev_kotor,
       COALESCE(r.qty_retur, 0)                           AS qty_retur,
       ROUND(COALESCE(r.nilai_retur, 0), 0)               AS nilai_retur,
       ROUND(j.rev_kotor - COALESCE(r.nilai_retur, 0), 0) AS rev_bersih
FROM jual j
LEFT JOIN ret r ON j.produk = r.produk
ORDER BY rev_bersih DESC
LIMIT 50
""")

# 5. Produk dengan retur tertinggi
simpan("retur_produk", """
WITH jual AS (
    SELECT Description AS produk, SUM(Quantity) AS qty_jual
    FROM sales GROUP BY Description
    HAVING SUM(Quantity) >= 500
),
ret AS (
    SELECT Desc_up AS produk, SUM(qty_retur) AS qty_retur, COUNT(*) AS kali_retur
    FROM retur GROUP BY Desc_up
)
SELECT j.produk, j.qty_jual, r.qty_retur,
       ROUND(r.qty_retur * 100.0 / j.qty_jual, 1) AS retur_pct,
       r.kali_retur
FROM jual j JOIN ret r ON j.produk = r.produk
ORDER BY retur_pct DESC
LIMIT 20
""")

# 6. Ringkasan segmen RFM
simpan("rfm_ringkasan", """
SELECT segmen,
       COUNT(*)                    AS pelanggan,
       ROUND(SUM(monetary), 0)     AS revenue,
       ROUND(AVG(recency), 0)      AS rata2_recency,
       ROUND(AVG(frequency), 1)    AS rata2_frekuensi
FROM rfm_segmen
GROUP BY segmen
ORDER BY revenue DESC
""")

# 7. Detail RFM per pelanggan
simpan("rfm_pelanggan",
       "SELECT CustomerID, recency, frequency, monetary, r_skor, f_skor, m_skor, segmen FROM rfm_segmen")

# Unduh semua file
for f in sorted(os.listdir("export")):
    files.download(f"export/{f}")

kpi_ringkasan             1 baris
tren_bulanan             13 baris
revenue_negara           38 baris
produk_bersih            50 baris
retur_produk             20 baris
rfm_ringkasan             6 baris
rfm_pelanggan         4,334 baris


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (5 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# =====================================================
# CELL 10: SIMPAN QUERY SEBAGAI FILE .SQL
# =====================================================
VIEW_RETUR = """
DROP VIEW IF EXISTS retur;
CREATE VIEW retur AS
SELECT *, UPPER(TRIM(Description)) AS Desc_up,
       -Quantity AS qty_retur, -Quantity * UnitPrice AS nilai_retur
FROM cancel
WHERE StockCode NOT IN ('POST','D','M','DOT','CRUK','BANK CHARGES','AMAZONFEE','S','B','PADS');
"""

VIEW_RFM = """
DROP VIEW IF EXISTS rfm_base;
DROP VIEW IF EXISTS rfm_skor;
DROP VIEW IF EXISTS rfm_segmen;

CREATE VIEW rfm_base AS
SELECT CustomerID,
    CAST(julianday((SELECT DATE(MAX(InvoiceDate), '+1 day') FROM sales))
         - julianday(DATE(MAX(InvoiceDate))) AS INTEGER) AS recency,
    COUNT(DISTINCT InvoiceNo) AS frequency,
    ROUND(SUM(Revenue), 2)    AS monetary
FROM sales
WHERE punya_customer_id = 1
GROUP BY CustomerID;

CREATE VIEW rfm_skor AS
SELECT *,
    NTILE(5) OVER (ORDER BY recency DESC)        AS r_skor,
    NTILE(5) OVER (ORDER BY frequency, monetary) AS f_skor,
    NTILE(5) OVER (ORDER BY monetary)            AS m_skor
FROM rfm_base;

CREATE VIEW rfm_segmen AS
SELECT *,
    CASE
        WHEN r_skor >= 4 AND f_skor >= 4 THEN 'Champions'
        WHEN r_skor >= 3 AND f_skor >= 3 THEN 'Loyal'
        WHEN r_skor >= 4 AND f_skor <= 2 THEN 'Pelanggan Baru'
        WHEN r_skor <= 2 AND f_skor >= 3 THEN 'Berisiko Pergi'
        WHEN r_skor <= 2 AND f_skor <= 2 THEN 'Hilang'
        ELSE 'Perlu Perhatian'
    END AS segmen
FROM rfm_skor;
"""

os.makedirs("sql_files", exist_ok=True)

isi_sql = {
    "00_views_retur_rfm.sql":  ("Membuat view retur bersih dan view RFM", VIEW_RETUR + "\n" + VIEW_RFM),
    "01_tren_bulanan.sql":     ("Pertanyaan 1: tren penjualan per bulan", q1),
    "02a_revenue_negara.sql":  ("Pertanyaan 2a: negara dengan revenue terbesar", q2a),
    "02b_produk_terlaris.sql": ("Pertanyaan 2b: produk teratas berdasarkan revenue bersih", q2b),
    "03_segmentasi_rfm.sql":   ("Pertanyaan 3: ringkasan segmen pelanggan RFM", q3),
    "04_analisis_retur.sql":   ("Pertanyaan 4: produk dengan persentase retur tertinggi", q4),
}

for nama, (judul, isi) in isi_sql.items():
    with open(f"sql_files/{nama}", "w", encoding="utf-8") as f:
        f.write(f"-- {judul}\n-- Dataset: Online Retail (UCI), tabel: sales\n\n{isi.strip()}\n")
    print("Tersimpan:", nama)
    files.download(f"sql_files/{nama}")

Tersimpan: 00_views_retur_rfm.sql


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tersimpan: 01_tren_bulanan.sql


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (8 outputs left out)

<IPython.core.display.Javascript object>

Tersimpan: 04_analisis_retur.sql


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>